# Comparison Report — Joint 6-D Diffusion vs the RePaint-part Pipeline

Merges the outputs of the two systems and produces the head-to-head result.

| | System A | System B |
|---|---|---|
| notebook | `Joint6D_Completion_kaggle.ipynb` | `Pipeline_RePaintPart_kaggle.ipynb` |
| geometry | diffused jointly with colour, trained from scratch | frozen PoinTr (ShapeNet-55 checkpoint) |
| colour | diffused jointly with geometry | separate DDPM, RePaint-inpainted afterwards |
| part labels | **none** | PointNet part-seg, used by 4 of 6 methods |
| sees the mask in training | yes | no — only at inference |

## What this notebook will and will not do

It **refuses to merge** two runs whose shared-spec hash or config hash differ. That check is not
ceremony: it is the only thing standing between a real comparison and two sets of numbers that
merely look comparable. Same objects, same split, same frame, same metric code, or no table.

It also **separates the geometry claim from the colour claim**, because only one of them is fair.

## §1 · Locate the two runs

In [ ]:
import os, json, glob, hashlib, warnings, itertools
from pathlib import Path
warnings.filterwarnings("ignore", category=UserWarning)
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats

# Point these at the two runs. On Kaggle, attach both outputs as datasets and the glob finds them.
SEARCH = ["results", "results_pipeline",
          "/kaggle/input/*/results", "/kaggle/input/*/results_pipeline",
          "/kaggle/working/results", "/kaggle/working/results_pipeline"]
N_BOOT = 5000

def find_runs():
    found = {}
    for pat in SEARCH:
        for d in sorted(glob.glob(pat)):
            man, ev = Path(d) / "run_manifest.json", Path(d) / "tables/eval_raw.csv"
            if man.exists() and ev.exists():
                m = json.load(open(man))
                found.setdefault(m["system"], (Path(d), m, ev))
    return found


RUNS = find_runs()
print(f"found {len(RUNS)} run(s):\n")
for sysname, (d, m, ev) in RUNS.items():
    print(f"  {sysname:16s} {d}")
    print(f"  {'':16s} spec {m['spec_hash']}  config {m['cfg_hash']}")
if len(RUNS) < 2:
    print("\nNeed BOTH runs. Attach the other notebook's output directory and re-run.")

## §2 · The compatibility gate

Two hashes must match. `spec_hash` covers the loader, the split, the partial-frame normalisation
and every metric; `cfg_hash` covers category, difficulty, point count, split sizes, seed and the two
thresholds. If either differs, the runs are not measuring the same thing and the notebook stops
here — a diff of the offending config keys is printed so the mismatch is easy to fix.

In [ ]:
COMPATIBLE = False
if len(RUNS) >= 2:
    names = list(RUNS)
    specs = {n: RUNS[n][1]["spec_hash"] for n in names}
    cfgs = {n: RUNS[n][1]["cfg_hash"] for n in names}
    print(f"{'system':18} {'spec hash':>18} {'config hash':>18}")
    for n in names:
        print(f"{n:18} {specs[n]:>18} {cfgs[n]:>18}")

    ok_spec = len(set(specs.values())) == 1
    ok_cfg = len(set(cfgs.values())) == 1
    print(f"\nspec  : {'MATCH' if ok_spec else 'MISMATCH'}")
    print(f"config: {'MATCH' if ok_cfg else 'MISMATCH'}")

    if not ok_cfg:
        a, b = RUNS[names[0]][1]["cfg"], RUNS[names[1]][1]["cfg"]
        print("\ndiffering config keys:")
        for k in sorted(set(a) | set(b)):
            if a.get(k) != b.get(k):
                print(f"  {k:18} {names[0]}={a.get(k)!r}   {names[1]}={b.get(k)!r}")
    if not ok_spec:
        print("\nThe shared spec text differs between the notebooks. Re-copy the spec cell from one")
        print("into the other so both embed byte-identical code, then re-run both.")

    COMPATIBLE = ok_spec and ok_cfg
    print("\n" + ("-> COMPARABLE. Proceeding." if COMPATIBLE else
                  "-> NOT COMPARABLE. The tables below are suppressed on purpose."))

## §3 · Merge

In [ ]:
if COMPATIBLE:
    frames = []
    for n, (d, m, ev) in RUNS.items():
        df = pd.read_csv(ev)
        df["system"] = n
        frames.append(df)
    ALL = pd.concat(frames, ignore_index=True)
    ALL = ALL[ALL.region == "missing"]
    METRICS = [c for c in ALL.columns if c in
               ["cd_l1", "cd_l2", "fscore", "dE00_sym", "dE00_gt2pred", "dE00_pred2gt",
                "dE00_matched", "match_rate", "colour_swd", "region_dE", "colour_edge_iou"]]
    DIR = {"cd_l1": "lower", "cd_l2": "lower", "fscore": "higher", "dE00_sym": "lower",
           "dE00_gt2pred": "lower", "dE00_pred2gt": "lower", "dE00_matched": "lower",
           "match_rate": "higher", "colour_swd": "lower", "region_dE": "lower",
           "colour_edge_iou": "higher"}
    GEOM = ["cd_l1", "cd_l2", "fscore"]
    COLOUR = [m for m in METRICS if m not in GEOM]

    common = set.intersection(*[set(g.mid) for _, g in ALL.groupby("system")])
    ALL = ALL[ALL.mid.isin(common)]
    print(f"{len(ALL)} rows | {ALL.system.nunique()} systems | "
          f"{ALL.method.nunique()} methods | {len(common)} objects scored by BOTH")
    print(f"\nmethods per system:")
    for s, g in ALL.groupby("system"):
        print(f"  {s:16s} {sorted(g.method.unique())}")
    print(f"\npredicted point counts (after EQUALIZE_PRED_N):")
    print(ALL.groupby(["system", "method"])[["n_pred", "n_gt"]].mean().round(1).to_string())
    ALL.to_csv("merged_eval.csv", index=False)

In [ ]:
if COMPATIBLE:
    TABLE = ALL.groupby(["system", "method"])[METRICS].mean()
    print("=" * 100)
    print("EVERY METHOD, BOTH SYSTEMS — missing region")
    print("=" * 100)
    print(TABLE.round(4).to_string())

    print("\n\nBEST METHOD PER SYSTEM (by dE00_sym, the symmetric colour error):")
    best = {}
    for s, g in ALL.groupby("system"):
        b = g.groupby("method").dE00_sym.mean().idxmin()
        best[s] = b
        print(f"  {s:16s} {b}")

## §4 · Head-to-head

Paired **per object** — both systems saw the same 40 objects, so the comparison uses each object as
its own control rather than comparing two independent means. Bootstrap CIs over objects, plus a
Wilcoxon signed-rank test.

Geometry and colour are reported separately, and only one of the two is a fair fight.

In [ ]:
def paired_systems(df, sa, ma, sb, mb, n_boot=N_BOOT):
    out = []
    A = df[(df.system == sa) & (df.method == ma)].groupby("mid")[METRICS].mean()
    B = df[(df.system == sb) & (df.method == mb)].groupby("mid")[METRICS].mean()
    idx = A.index.intersection(B.index)
    A, B = A.loc[idx], B.loc[idx]
    rng = np.random.default_rng(0)
    boot_idx = rng.integers(0, len(idx), (n_boot, len(idx)))
    for met in METRICS:
        a, b = A[met].values, B[met].values
        keep = np.isfinite(a) & np.isfinite(b)
        a, b = a[keep], b[keep]
        if len(a) < 5:
            continue
        sign = -1 if DIR[met] == "lower" else 1
        denom = abs(a.mean())
        scale = 100.0 / denom if denom > 1e-12 else 1.0
        bi = boot_idx[:, :len(a)] % len(a)
        boot = sign * (b[bi].mean(1) - a[bi].mean(1)) * scale
        lo, hi = np.percentile(boot, [2.5, 97.5])
        try:
            p = stats.wilcoxon(a, b).pvalue
        except Exception:
            p = np.nan
        out.append(dict(metric=met, kind="geometry" if met in GEOM else "colour",
                        A=a.mean(), B=b.mean(), improvement_B_over_A=sign * (b.mean()-a.mean())*scale,
                        ci_lo=lo, ci_hi=hi, win_rate_B=float((sign * (b - a) > 0).mean()),
                        wilcoxon_p=p,
                        verdict="B wins" if lo > 0 else ("A wins" if hi < 0 else "unresolved")))
    return pd.DataFrame(out)


if COMPATIBLE and len(best) >= 2:
    sa, sb = "joint6d", "repaint_part"
    if sa in best and sb in best:
        H2H = paired_systems(ALL, sa, best[sa], sb, best[sb])
        print("=" * 104)
        print(f"HEAD TO HEAD   A = {sa} / {best[sa]}     B = {sb} / {best[sb]}")
        print("improvement_B_over_A > 0 means the PIPELINE (B) is better")
        print("=" * 104)
        for kind in ["geometry", "colour"]:
            sub = H2H[H2H.kind == kind]
            if not len(sub): continue
            print(f"\n--- {kind.upper()} ---")
            print(sub[["metric", "A", "B", "improvement_B_over_A", "ci_lo", "ci_hi",
                       "win_rate_B", "wilcoxon_p", "verdict"]].round(4).to_string(index=False))
        H2H.to_csv("head_to_head.csv", index=False)
        print("""
NOTE ON THE GEOMETRY BLOCK: System B's geometry comes from a PoinTr checkpoint trained on
ShapeNet-55, which contains these categories and plausibly these exact models. System A trains
from scratch on 150 shapes. The geometry rows are therefore NOT a fair comparison and should be
read as an optimistic bound on B. The COLOUR rows are the meaningful ones — there both systems
generate from scratch.""")

In [ ]:
if COMPATIBLE:
    fig, axes = plt.subplots(2, 3, figsize=(16, 7.5))
    show = ["cd_l1", "fscore", "dE00_sym", "colour_swd", "region_dE", "colour_edge_iou"]
    order = TABLE.reset_index()
    for ax, met in zip(axes.ravel(), show):
        labels, vals, errs, cols = [], [], [], []
        for (s, m), row in TABLE.iterrows():
            per = ALL[(ALL.system == s) & (ALL.method == m)].groupby("mid")[met].mean().dropna()
            if len(per) < 3: continue
            rng = np.random.default_rng(0)
            bs = per.values[rng.integers(0, len(per), (2000, len(per)))].mean(1)
            labels.append(f"{m}")
            vals.append(per.mean())
            errs.append([per.mean() - np.percentile(bs, 2.5), np.percentile(bs, 97.5) - per.mean()])
            cols.append("#2f6f4f" if s == "joint6d" else "#b1622c")
        y = np.arange(len(labels))
        ax.barh(y, vals, xerr=np.array(errs).T, capsize=3, color=cols)
        ax.set_yticks(y); ax.set_yticklabels(labels, fontsize=7)
        ax.invert_yaxis()
        ax.set_title(f"{met}  ({'lower' if DIR[met]=='lower' else 'higher'} better)", fontsize=9)
        ax.grid(alpha=.25, axis="x")
    import matplotlib.patches as mp
    fig.legend(handles=[mp.Patch(color="#2f6f4f", label="A · joint6d"),
                        mp.Patch(color="#b1622c", label="B · repaint_part")],
               loc="lower center", ncol=2, fontsize=9)
    fig.suptitle("All methods, both systems — missing region, 95% bootstrap CI over objects",
                 fontsize=12)
    fig.tight_layout(rect=[0, .04, 1, 1])
    fig.savefig("comparison_all_methods.png", dpi=140, bbox_inches="tight"); plt.close(fig)
    print("saved comparison_all_methods.png")

    # per-object scatter: where does each system win?
    if "H2H" in dir():
        fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
        for ax, met in zip(axes, ["cd_l1", "dE00_sym", "colour_swd"]):
            A = ALL[(ALL.system == sa) & (ALL.method == best[sa])].groupby("mid")[met].mean()
            B = ALL[(ALL.system == sb) & (ALL.method == best[sb])].groupby("mid")[met].mean()
            i = A.index.intersection(B.index)
            ax.scatter(A.loc[i], B.loc[i], s=20, alpha=.75)
            lim = [min(A.loc[i].min(), B.loc[i].min()), max(A.loc[i].max(), B.loc[i].max())]
            ax.plot(lim, lim, "k--", lw=1)
            ax.set_xlabel(f"A · joint6d"); ax.set_ylabel(f"B · repaint_part")
            ax.set_title(f"{met} (lower better)\npoints below the line = B better", fontsize=9)
            ax.grid(alpha=.3)
        fig.suptitle("Per-object, paired", fontsize=11)
        fig.tight_layout(); fig.savefig("comparison_per_object.png", dpi=140); plt.close(fig)
        print("saved comparison_per_object.png")

## §5 · Report

In [ ]:
def md_table(df, index=False):
    try:
        return df.to_markdown(index=index)
    except Exception:
        d = df.reset_index() if index else df
        cols = [str(c) for c in d.columns]
        f = lambda v: ("nan" if v != v else f"{v:.4g}") if isinstance(v, float) else str(v)
        return "\n".join(["| " + " | ".join(cols) + " |",
                          "|" + "|".join("---" for _ in cols) + "|"]
                         + ["| " + " | ".join(f(v) for v in r) + " |"
                            for r in d.itertuples(index=False)])


L = ["# Colored Point Cloud Completion — System Comparison", ""]
if not COMPATIBLE:
    L += ["**The two runs are not comparable.** See §2 for the differing hashes. No table is",
          "produced, deliberately: publishing numbers from mismatched runs is worse than",
          "publishing none."]
else:
    cfg = list(RUNS.values())[0][1]["cfg"]
    L += [f"*Shared spec `{list(RUNS.values())[0][1]['spec_hash']}` · "
          f"config `{list(RUNS.values())[0][1]['cfg_hash']}` — verified identical across both runs.*",
          "",
          f"- category **{cfg['category']}**, difficulties {cfg['difficulties']}, "
          f"{cfg['num_points']} points",
          f"- {cfg['num_train']} train / {cfg['num_val']} val, seed {cfg['seed']}, "
          f"{len(common)} objects scored by both",
          f"- predictions equalised to `{cfg['equalize_pred_n']}` points before scoring", "",
          "## 1. All methods", "", md_table(TABLE.round(4), index=True), ""]
    if "H2H" in dir():
        for kind in ["colour", "geometry"]:
            sub = H2H[H2H.kind == kind]
            if not len(sub): continue
            L += [f"## 2. Head to head — {kind}", "",
                  f"A = `{sa}/{best[sa]}`, B = `{sb}/{best[sb]}`. Positive = B better.", "",
                  md_table(sub[["metric", "A", "B", "improvement_B_over_A", "ci_lo", "ci_hi",
                                "win_rate_B", "wilcoxon_p", "verdict"]].round(4)), ""]
        cw = H2H[(H2H.kind == "colour")]
        aw, bw = (cw.verdict == "A wins").sum(), (cw.verdict == "B wins").sum()
        L += ["## 3. Reading", "",
              f"On **colour** — the fair half of this comparison — System A wins {aw} metrics, "
              f"System B wins {bw}, and {len(cw)-aw-bw} are unresolved at this sample size.", "",
              "**The geometry block is not a fair fight.** System B's geometry comes from a PoinTr "
              "checkpoint trained on ShapeNet-55, which contains these categories and plausibly "
              "these exact models; System A trains from scratch on "
              f"{cfg['num_train']} shapes. Read B's geometry numbers as an optimistic bound, not "
              "as a result.", "",
              "Two further asymmetries, neither of which is a bug: System B's colour DDPM never "
              "sees an occlusion mask during training (it meets one only at inference, via "
              "RePaint), while System A trains with the mask in the loop; and the two systems "
              "natively emit different numbers of points, which is why `EQUALIZE_PRED_N` exists "
              "and why `n_pred` is reported.", "",
              "## 4. Limits", "",
              "1. **One training seed per system.** Every interval here is over objects, not over "
              "training runs. A few-percent gap is unmeasured, not absent.",
              f"2. One category, {len(common)} objects, {len(METRICS)} uncorrected metric tests.",
              "3. No external baseline (PoinTr-as-published, SeedFormer, AdaPoinTr) — all claims "
              "are internal to these two systems.", ""]

rep = "\n".join(L)
open("comparison_report.md", "w").write(rep)
print(rep[:3000])
print(f"\n... full report in comparison_report.md ({len(rep)} chars)")